In [ ]:
import pandas as pd
import numpy as np
import gc
import os

INPUT_DATA_PATH = "/kaggle/input/notebooks/luminhanh/ba-model-prep-data/" 

print("🚀 Đang nạp dữ liệu đã tiền xử lý...")

# 1. Nạp Features
X_train = pd.read_parquet(INPUT_DATA_PATH + 'X_train.parquet')
X_val = pd.read_parquet(INPUT_DATA_PATH + 'X_val.parquet')
X_test = pd.read_parquet(INPUT_DATA_PATH + 'X_test.parquet')

# 2. Nạp Target (Chuyển ngay sang Numpy để sẵn sàng cho Model)
y_train = pd.read_parquet(INPUT_DATA_PATH + 'y_train.parquet').values
y_val = pd.read_parquet(INPUT_DATA_PATH + 'y_val.parquet').values

# 3. Nạp các thông tin bổ trợ cho hậu xử lý
test_id_matrix = pd.read_pickle(INPUT_DATA_PATH + 'test_id_matrix.pkl')

print(f"✅ Nạp xong! Kích thước X_train: {X_train.shape}")
gc.collect()

🚀 Đang nạp dữ liệu đã tiền xử lý...
✅ Nạp xong! Kích thước X_train: (1005090, 633)


0

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
import gc
import os

print("🚀 Chuẩn bị dữ liệu và Huấn luyện 16 Mô hình CATBOOST...")

DATA_PATH = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/' 

# --- 1. NẠP DỮ LIỆU TỪ PARQUET ---
if 'X_train' not in locals():
    print("⏳ Đang nạp dữ liệu từ các file .parquet...")
    X_train = pd.read_parquet(DATA_PATH + 'X_train.parquet')
    y_train = pd.read_parquet(DATA_PATH + 'y_train.parquet').values
    X_val = pd.read_parquet(DATA_PATH + 'X_val.parquet')
    y_val = pd.read_parquet(DATA_PATH + 'y_val.parquet').values
    X_test = pd.read_parquet(DATA_PATH + 'X_test.parquet')
    print("✅ Đã nạp xong dữ liệu!")
else:
    print("✅ Dữ liệu đã có sẵn trong RAM!")

# --- 2. XỬ LÝ CATEGORICAL CHO CATBOOST ---
cat_features = ['store_nbr', 'item_nbr', 'family', 'class', 'city', 'state', 'type', 'cluster']

for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)
        X_test[col] = X_test[col].astype(str)

# --- 3. CẮT TỈA KHỚP DÒNG & TRỌNG SỐ ---
train_size = min(X_train.shape[0], y_train.shape[0])
val_size = min(X_val.shape[0], y_val.shape[0])

X_train_fix = X_train.iloc[:train_size].copy()
y_train_fix = np.array(y_train)[:train_size]
X_val_fix = X_val.iloc[:val_size].copy()
y_val_fix = np.array(y_val)[:val_size]

if 'perishable' in X_train_fix.columns:
    train_weights = X_train_fix['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0).values
    val_weights = X_val_fix['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0).values
    
    X_train_model = X_train_fix.drop(columns=['perishable'])
    X_val_model = X_val_fix.drop(columns=['perishable'])
    X_test_model = X_test.drop(columns=['perishable'])
else:
    train_weights = None
    val_weights = None
    X_train_model = X_train_fix
    X_val_model = X_val_fix
    X_test_model = X_test.copy()

del X_train_fix, X_val_fix
gc.collect()

# --- 4. CẤU HÌNH THAM SỐ CATBOOST (TỪ OPTUNA) ---
cb_params = {
    'iterations': 1500,              
    'learning_rate': 0.059234900214243276,
    'l2_leaf_reg': 3.8258800788689413,
    'random_strength': 4.353721638364459,
    'bagging_temperature': 0.21782504502987693,
    'border_count': 128,
    'depth': 9,                      
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'thread_count': 4,               
    'task_type': 'GPU'   
}

actual_cat_features = [c for c in cat_features if c in X_train_model.columns]

# --- 5. TIẾN HÀNH HUẤN LUYỆN 16 NGÀY ---
test_preds = []

for i in range(16):
    print(f"\n--- Đang huấn luyện cho Ngày {i+1} ---")
    
    train_pool = Pool(
        data=X_train_model, 
        label=y_train_fix[:, i], 
        weight=train_weights, 
        cat_features=actual_cat_features
    )
    
    val_pool = Pool(
        data=X_val_model, 
        label=y_val_fix[:, i], 
        weight=val_weights, 
        cat_features=actual_cat_features
    )
    
    model = CatBoostRegressor(**cb_params)
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
        verbose=200,                
        use_best_model=True
    )
    
    test_preds.append(model.predict(X_test_model))
    
    del train_pool, val_pool, model
    gc.collect()

print("\n✅ HOÀN TẤT HUẤN LUYỆN 16 MODELS CATBOOST!")

🚀 Chuẩn bị dữ liệu và Huấn luyện 16 Mô hình CATBOOST...
✅ Dữ liệu đã có sẵn trong RAM!

--- Đang huấn luyện cho Ngày 1 ---
0:	learn: 1.0113906	test: 0.9975820	best: 0.9975820 (0)	total: 335ms	remaining: 8m 22s
200:	learn: 0.5340726	test: 0.5338456	best: 0.5338456 (200)	total: 37s	remaining: 3m 59s
400:	learn: 0.5246788	test: 0.5302551	best: 0.5302551 (400)	total: 1m 12s	remaining: 3m 18s
600:	learn: 0.5179838	test: 0.5293464	best: 0.5293464 (600)	total: 1m 47s	remaining: 2m 40s
800:	learn: 0.5129079	test: 0.5289590	best: 0.5289512 (799)	total: 2m 21s	remaining: 2m 3s
1000:	learn: 0.5084931	test: 0.5287194	best: 0.5287194 (1000)	total: 2m 57s	remaining: 1m 28s
1200:	learn: 0.5042296	test: 0.5286055	best: 0.5285890 (1177)	total: 3m 32s	remaining: 52.8s
bestTest = 0.5285377422
bestIteration = 1285
Shrink model to first 1286 iterations.

--- Đang huấn luyện cho Ngày 2 ---
0:	learn: 0.9626072	test: 0.9593836	best: 0.9593836 (0)	total: 218ms	remaining: 5m 27s
200:	learn: 0.5538515	test: 0.56

In [3]:
import pandas as pd
import numpy as np
import gc
import os

print("🚀 Hậu xử lý và Xuất file nộp bài...")

# --- TỰ ĐỘNG TÌM ĐƯỜNG DẪN DỮ LIỆU ---
PATH = ""
for dirname, _, filenames in os.walk('/kaggle/input/notebooks/luminhanh/ba-model-prep-data'):
    if 'test.csv' in filenames or 'test.csv.7z' in filenames:
        PATH = dirname + '/'
        break

if PATH == "":
    print("❌ CẢNH BÁO: Không tìm thấy file test.csv trong /kaggle/input!")
else:
    print(f"📂 Tìm thấy thư mục dữ liệu tại: {PATH}")

# 1. GIẢI PHÓNG BỘ NHỚ TRIỆT ĐỂ
for var in ['X_train', 'y_train', 'X_val', 'y_val', 'X_train_model', 'X_val_model', 'train_pool', 'val_pool', 'dtrain', 'dval']:
    if var in locals():
        del globals()[var]
gc.collect()

# 2. XỬ LÝ MẢNG DỰ ĐOÁN
print("   -> Đang chuyển đổi ma trận dự đoán (16 ngày)...")
y_test_pred = np.array(test_preds, dtype=np.float32).transpose()

if 'test_preds' in locals():
    del test_preds
gc.collect()

y_test_pred = np.clip(np.expm1(y_test_pred), 0, None)

# 3. TẠO DATAFRAME KẾT QUẢ TỪ X_TEST
df_preds = pd.DataFrame(
    y_test_pred, 
    index=X_test.index, 
    columns=pd.date_range("2017-08-16", periods=16)
)
for col in df_preds.columns:
    df_preds[col] = df_preds[col].astype(np.float32)

del y_test_pred
gc.collect()

# 4. BIẾN ĐỔI SANG DẠNG DỌC VÀ FIX KIỂU DỮ LIỆU
print("   -> Đang biến đổi dữ liệu sang dạng Dọc và Đồng bộ Index...")

if 'store_nbr' in df_preds.columns or 'store_nbr' in X_test.columns:
    df_preds['store_nbr'] = X_test['store_nbr'].values.astype(np.int32)
    df_preds['item_nbr'] = X_test['item_nbr'].values.astype(np.int32)
    df_preds = df_preds.set_index(['store_nbr', 'item_nbr']).stack().to_frame("unit_sales")
else:
    df_preds = df_preds.stack().to_frame("unit_sales")
    df_preds.index = df_preds.index.set_levels([
        df_preds.index.levels[0].astype(np.int32),
        df_preds.index.levels[1].astype(np.int32),
        df_preds.index.levels[2]
    ])

df_preds.index.names = ["store_nbr", "item_nbr", "date"]

# 5. ĐỌC FILE TEST GỐC ĐỂ LẤY CỘT ID
print("   -> Đang map ID từ file test.csv...")
df_test_kaggle = pd.read_csv(f'{PATH}test.csv', 
                             usecols=['id', 'store_nbr', 'item_nbr', 'date'],
                             parse_dates=['date'])
df_test_kaggle.set_index(['store_nbr', 'item_nbr', 'date'], inplace=True)

# Join kết quả dự đoán với ID của Kaggle
submission = df_test_kaggle.join(df_preds)

del df_test_kaggle, df_preds, X_test
gc.collect()

# 6. HOÀN THIỆN VÀ XUẤT FILE CSV
submission['unit_sales'] = submission['unit_sales'].fillna(0).astype(np.float32)

OUTPUT_FILE = 'submission_final_fixed.csv'
submission[['id', 'unit_sales']].to_csv(OUTPUT_FILE, index=False, float_format='%.4f')

print(f"🏆 THÀNH CÔNG! File đã được tạo tại: {OUTPUT_FILE}")

🚀 Hậu xử lý và Xuất file nộp bài...
📂 Tìm thấy thư mục dữ liệu tại: /kaggle/input/notebooks/luminhanh/ba-model-prep-data/
   -> Đang chuyển đổi ma trận dự đoán (16 ngày)...
   -> Đang biến đổi dữ liệu sang dạng Dọc và Đồng bộ Index...
   -> Đang map ID từ file test.csv...
🏆 THÀNH CÔNG! File đã được tạo tại: submission_final_fixed.csv
